# Development of basic steering vector experiment

Aim of notebook is as dirty place to develop the code. Includes various things being printed, and experiments with gpu memory usage and what batch size I can use.

In [1]:
from data_utils import generate_training_prompts

In [2]:
import pandas as pd
from sklearn.model_selection import KFold

def get_cv_splits(data: pd.DataFrame, n_splits: int = 5, random_state: int = 42) -> list[tuple[pd.DataFrame, pd.DataFrame]]:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    cv_splits = []
    for train_idx, test_idx in kf.split(data):
        train_df = data.iloc[train_idx].copy()
        test_df = data.iloc[test_idx].copy()
        cv_splits.append((train_df, test_df))
    
    return cv_splits

path_to_data = "antonyms.json"
data = pd.read_json(path_to_data)
cv_pairs = get_cv_splits(data, n_splits=5, random_state=42)
data_train, data_test = cv_pairs[0]


In [3]:
data_train.head(10)

,input,target
0,flawed,perfect
1,orthodox,unorthodox
2,true,false
3,daily,nightly
4,distribution,concentration
5,valid,invalid
6,expand,contract
7,practical,impractical
8,privilege,disadvantage
9,mammoth,tiny


In [4]:
data_test.head()

,input,target
18,proceed,halt
25,privacy,publicity
29,professional,amateur
43,fascism,democracy
44,super,inferior


In [5]:
n_pairs_per_training_prompt = 10
training_prompts = generate_training_prompts(data_train, n_pairs_per_training_prompt, separator=', ')
print(training_prompts[0])

flawed:perfect, orthodox:unorthodox, true:false, daily:nightly, distribution:concentration, valid:invalid, expand:contract, practical:impractical, privilege:disadvantage, mammoth:


In [6]:
import torch
from transformer_lens import HookedTransformer

MODEL_NAME = "EleutherAI/gpt-j-6b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16"

model = HookedTransformer.from_pretrained_no_processing(
    model_name=MODEL_NAME, device=DEVICE, dtype=DTYPE
)
model.eval()


/root/miniconda/envs/conceptors/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bi

Loaded pretrained model EleutherAI/gpt-j-6b into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [7]:
def check_torch_gpu_memory():
    # Total memory available on the GPU
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # in GB
    
    # Memory reserved by PyTorch's allocator
    reserved_memory = torch.cuda.memory_reserved() / 1e9  # in GB
    
    # Memory actually allocated by PyTorch (subset of reserved)
    allocated_memory = torch.cuda.memory_allocated() / 1e9  # in GB
    
    print(f"Total GPU Memory: {total_memory:.4f} GB")
    print(f"Reserved by PyTorch: {reserved_memory:.4f} GB")
    print(f"Allocated by PyTorch: {allocated_memory:.4f} GB")

In [8]:
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 12.2453 GB
Allocated by PyTorch: 12.2358 GB


## checking gpu memory while using the model

In [9]:
model("What is capital of France?")
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.1624 GB
Allocated by PyTorch: 12.2443 GB


In [10]:
model("What is capital of UK?")
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.1624 GB
Allocated by PyTorch: 12.2443 GB


In [11]:
torch.cuda.empty_cache()
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 12.2662 GB
Allocated by PyTorch: 12.2443 GB


In [12]:
extraction_layers = [9]
prompts = ["What is capital of France?", "What is capital of UK?"]

names = [f"blocks.{layer}.hook_resid_pre" for layer in extraction_layers]
cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n in names)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 12.2662 GB
Allocated by PyTorch: 12.2443 GB


In [13]:
with model.hooks(fwd_hooks=caching_hooks):
    model.tokenizer.padding_side = "left"
    _ = model(prompts)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.2379 GB
Allocated by PyTorch: 15.2134 GB


In [14]:
torch.cuda.empty_cache()
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.2358 GB
Allocated by PyTorch: 15.2134 GB


In [22]:
del cache
torch.cuda.empty_cache()
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.2358 GB
Allocated by PyTorch: 15.2134 GB


In [40]:
# This function is commented out because I use a batched version below
# def extract_activations_last_token(
#         model: HookedTransformer,
#         prompts: list[str],
#         extraction_layers: list[int]
# ) -> dict[int, torch.Tensor]:
#     """
#     Extract activations for the last token of each steering prompt from specific layers of the model.

#     Parameters:
#     model (HookedTransformer): The model used for generating text.
#     prompts (list): List of prompts to extract activations for.
#     extraction_layers (list): The layers from which activations are extracted.
#     device (str): The computing device (e.g., 'cuda', 'cpu').

#     Returns:
#     dict: A dictionary where each key is a layer number and each value is the
#         activations for the last token of each prompt. Shape: (n_prompts, n_activations).
#     """
    # activations_dict = {}
    # names = [f"blocks.{layer}.hook_resid_pre" for layer in extraction_layers]
    # cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n in names)

    # with model.hooks(fwd_hooks=caching_hooks):
    #     model.tokenizer.padding_side = "left"
    #     _ = model(prompts)

    # for layer in extraction_layers:
    #     prompt_activations = cache[f"blocks.{layer}.hook_resid_pre"].detach().cpu()
    #     last_token_activations = prompt_activations[:, -1, :].squeeze()
    #     activations_tensor = torch.tensor(
    #         last_token_activations.numpy(), dtype=torch.float, device='cpu'
    #     )
    #     activations_dict[layer] = activations_tensor

#     return activations_dict


# this one uses batches
def extract_activations_last_token(
        model: HookedTransformer,
        prompts: list[str],
        extraction_layers: list[int],
        batch_size: int = 64,
) -> dict[int, torch.Tensor]:
    """
    Extract activations for the last token of each steering prompt from specific layers of the model.
    Processes prompts in batches to avoid memory issues.

    If you get an error about memory, try reducing the batch size.

    Parameters:
    model (HookedTransformer): The model used for generating text.
    prompts (list): List of prompts to extract activations for.
    extraction_layers (list): The layers from which activations are extracted.
    batch_size (int): Number of prompts to process at once.

    Returns:
    dict: A dictionary where each key is a layer number and each value is the
        activations for the last token of each prompt. Shape: (n_prompts, n_activations).
    """
    activations_dict = {layer: [] for layer in extraction_layers}
    names = [f"blocks.{layer}.hook_resid_pre" for layer in extraction_layers]
    
    # Process prompts in batches
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n in names)
        
        with model.hooks(fwd_hooks=caching_hooks):
            model.tokenizer.padding_side = "left"
            _ = model(batch_prompts)
            
        for layer in extraction_layers:
            prompt_activations = cache[f"blocks.{layer}.hook_resid_pre"].detach().cpu()
            last_token_activations = prompt_activations[:, -1, :].squeeze()
            # Handle the case where there's only one prompt in the batch
            if len(batch_prompts) == 1:
                last_token_activations = last_token_activations.unsqueeze(0)
            activations_dict[layer].append(last_token_activations)
        
        # Clear CUDA cache after each batch
        torch.cuda.empty_cache()
    
    # Concatenate the batched results
    for layer in extraction_layers:
        activations_dict[layer] = torch.cat(activations_dict[layer], dim=0)
    
    return activations_dict

## investigate memory as you vary inputs

Main finding is that batch size is what controls the maximum amount of memory you can use.

In [36]:
extraction_layers = list(range(9,10))
activations_last_token = extract_activations_last_token(model, training_prompts[0:10], extraction_layers, batch_size=1)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 38.1933 GB
Allocated by PyTorch: 15.2134 GB


In [30]:
# This shows that the number of training prompts is not the issue
extraction_layers = list(range(9,10))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=1)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 18.6080 GB
Allocated by PyTorch: 15.2134 GB


In [31]:
# Indication that number of layers is not the issue
extraction_layers = list(range(9,13))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=1)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 18.6080 GB
Allocated by PyTorch: 15.2134 GB


In [32]:
# Indication that increasing batch size does increase amount of memory reserved!
extraction_layers = list(range(9,13))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=8)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 22.0872 GB
Allocated by PyTorch: 15.2134 GB


In [33]:
# Same again
extraction_layers = list(range(9,13))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=16)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 28.0578 GB
Allocated by PyTorch: 15.2134 GB


In [41]:
# Same again.
extraction_layers = list(range(9,13))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=32)

check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 37.8851 GB
Allocated by PyTorch: 15.2134 GB


In [42]:
torch.cuda.empty_cache()
check_torch_gpu_memory()

Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.2652 GB
Allocated by PyTorch: 15.2134 GB


## Continue with pipeline

In [43]:
extraction_layers = list(range(9,17))
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=32)

torch.cuda.empty_cache()
check_torch_gpu_memory()


Total GPU Memory: 47.7609 GB
Reserved by PyTorch: 15.2652 GB
Allocated by PyTorch: 15.2134 GB


In [44]:
print(activations_last_token.keys())
assert len(activations_last_token.keys()) == len(extraction_layers)

print(activations_last_token[extraction_layers[0]].shape)

for layer in extraction_layers:
    assert activations_last_token[layer].shape == (len(training_prompts), model.cfg.d_model)


dict_keys([9, 10, 11, 12, 13, 14, 15, 16])
torch.Size([191, 4096])


In [45]:
def average_activations(
    activations: dict[int, torch.Tensor],
) -> dict[int, torch.Tensor]:
    """
    Computes averaged activations for all layers, averaging over all prompts.

    Args:
    activations: A dictionary where each key is a layer number and each value is the
        activations for the last token of each prompt. Shape: (n_prompts, n_activations).

    Returns:
    averaged_activations: dictionary containing the averaged activations for each layer.
        Averaging over all prompts.
        Keys are layer_index and values are the averaged activations.
        Shape of values: (d_model,).
    """
    averaged_activations: dict[int, torch.Tensor] = {}

    for layer in activations.keys():
        # Extract the last-token activations of steering examples at the specified layer
        activation = activations[layer]
        # Compute the average activations
        avg_activation = torch.mean(activation, dim=0)
        # Store the average activations in the cache
        averaged_activations[layer] = avg_activation.detach().cpu()

    return averaged_activations

activations_last_token_averaged = average_activations(activations_last_token)
print(activations_last_token_averaged.keys())

print(activations_last_token_averaged[extraction_layers[0]].shape)
for layer in extraction_layers:
    assert activations_last_token_averaged[layer].shape == (model.cfg.d_model,)

dict_keys([9, 10, 11, 12, 13, 14, 15, 16])
torch.Size([4096])


In [46]:
import pandas as pd
def generate_test_prompts(
    data: pd.DataFrame,
) -> list[str]:
    r"""
    Generates test prompts from the data.

    Args:
        data (pd.DataFrame): The data to generate training prompts from.
            Must have 'input'

    Returns:
        list[str]: A list of test prompts.
            Each training prompt is a string of the form "input:"
    """
    test_prompts: list[str] = []
    for i in range(len(data)):
        test_prompts.append(f"{data.iloc[i]['input']}:")
    
    return test_prompts

test_prompts = generate_test_prompts(data_test)
test_prompts[0:5]

['proceed:', 'privacy:', 'professional:', 'fascism:', 'super:']

In [47]:
from typing import Callable
def generate_hook_addition(steering_vector: torch.Tensor, beta: float) -> Callable:
    """
    Generates a hook function to add a steering vector to the last token.

    Parameters:
    - steering_vector (torch.Tensor): Steering vector.
    - beta (float): Scaling factor.

    Returns:
    - function: Hook function for adding steering vector.
    """

    def last_token_steering_hook(resid_pre, hook):
        for i in range(resid_pre.shape[0]):
            current_token_index = resid_pre.shape[1] - 1
            resid_pre[i, current_token_index, :] += steering_vector.squeeze().to(resid_pre.device) * beta

    return last_token_steering_hook

def generate_text(model: HookedTransformer, prompts: list[str], hooks: list[Callable], max_new_tokens: int = 5, temperature: float = 0) -> list[str]:
    with model.hooks(fwd_hooks=hooks):
        # for prompt in prompts:
        #     result = model.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        #     results.append(result)
        #     torch.cuda.empty_cache()
        results = model.generate(prompts, max_new_tokens=max_new_tokens, temperature=temperature, padding_side='left', verbose=False)
    
    # remove the original prompts from the results
    results = [result[len(prompt):] for result, prompt in zip(results, prompts)]

    return results

In [51]:
layer = extraction_layers[0]
beta = 2
addition_hook = generate_hook_addition(steering_vector=activations_last_token_averaged[layer], beta=beta)
hooks = [
    (f"blocks.{layer}.hook_resid_pre", addition_hook)
]

predictions = generate_text(model, test_prompts, hooks)


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:01<00:00,  2.80it/s]


In [52]:
for test, prediction in zip(test_prompts[110:115], predictions[110:115]):
    print(f"{test}{prediction}")
    print('='*20)

gritty:grgrgrgrgr
swift:

swswsw
devil:devdevdevdevdev
clandestine:clclclclcl
hard:softsofthardhardhard


In [53]:
# function to calculate the accuracy of the predictions

def calculate_accuracy(predictions: list[str], targets: pd.Series) -> float:
    """
    Calculates the accuracy of the predictions, by asking if the prediction starts with the target.

    Args:
        predictions (list[str]): A list of predictions.
        targets (pd.Series): A series of targets. Typically the 'target' column of the test set dataframe.

    Returns:
        float: The accuracy of the predictions.
    """
    correct = 0
    for prediction, target in zip(predictions, targets):
        if prediction.startswith(target):
            correct += 1
    return correct / len(predictions)

calculate_accuracy(predictions, data_test['target'])

0.07291666666666667

## Loop through hyperparameters and splits

In [56]:
from tqdm import tqdm

split_numbers = list(range(5))
n_pairs_per_training_prompt = 10
extraction_layers = list(range(9,17))
betas = [0, 1, 2, 3,4,5]

# create a dataframe to store the results
results_df = pd.DataFrame(columns=['split', 'layer', 'beta', 'accuracy'])

# Create a progress bar for the total number of iterations
total_iterations = len(split_numbers) * len(extraction_layers) * len(betas)
pbar = tqdm(total=total_iterations, desc="Overall Progress")

for split_number in split_numbers:
    # create data
    data_train, data_test = cv_pairs[split_number]
    training_prompts = generate_training_prompts(data_train, n_pairs_per_training_prompt, separator=', ')
    test_prompts = generate_test_prompts(data_test)

    # train steering vector on training prompts
    print(f"Split {split_number}: Extracting activations...")
    activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=32)
    activations_last_token_averaged = average_activations(activations_last_token)
    for layer in extraction_layers:
        assert activations_last_token[layer].shape == (len(training_prompts), model.cfg.d_model)
        assert activations_last_token_averaged[layer].shape == (model.cfg.d_model,)
    
    # test steering vector on test prompts
    for layer in extraction_layers:
        layer_results = []
        for beta in betas:
            addition_hook = generate_hook_addition(steering_vector=activations_last_token_averaged[layer], beta=beta)
            hooks = [(f"blocks.{layer}.hook_resid_pre", addition_hook)]
            predictions = generate_text(model, test_prompts, hooks)
            accuracy = calculate_accuracy(predictions, data_test['target'])
            layer_results.append((beta, accuracy))
            print(f"Split {split_number}, Beta {beta}, Layer {layer}, Accuracy {accuracy}")
            if len(results_df) == 0:
                results_df = pd.DataFrame({
                    'split': [split_number], 
                    'layer': [layer], 
                    'beta': [beta], 
                    'accuracy': [accuracy]
                })
            else:
                results_df = pd.concat([results_df, pd.DataFrame({
                    'split': [split_number], 
                    'layer': [layer], 
                    'beta': [beta], 
                    'accuracy': [accuracy]
                })], ignore_index=True)
            pbar.update(1)

pbar.close()
print("All experiments completed!")


Overall Progress:  13%|█▎        | 21/160 [02:34<16:59,  7.33s/it]


Split 0: Extracting activations...


100%|██████████| 5/5 [00:01<00:00,  2.81it/s]



Split 0, Beta 0, Layer 9, Accuracy 0.0


100%|██████████| 5/5 [00:01<00:00,  2.82it/s]



...

Split 4, Beta 3, Layer 16, Accuracy 0.006263048016701462


100%|██████████| 5/5 [00:01<00:00,  2.76it/s]



Split 4, Beta 4, Layer 16, Accuracy 0.0020876826722338203


100%|██████████| 5/5 [00:01<00:00,  2.75it/s]

Overall Progress: 100%|██████████| 240/240 [07:45<00:00,  1.94s/it]

Split 4, Beta 5, Layer 16, Accuracy 0.0
All experiments completed!


In [62]:
# print the top 20 results, after aggregating over splits
results_df.groupby(['layer', 'beta'])['accuracy'].mean().sort_values(ascending=False)

layer  beta
11     3       0.116759
       4       0.114255
10     4       0.113007
       5       0.110088
       3       0.109249
11     2       0.102162
       5       0.102158
13     2       0.100071
12     3       0.100070
       2       0.095905
13     3       0.094650
10     2       0.093403
14     2       0.080471
12     4       0.075470
9      3       0.068392
       4       0.068389
14     3       0.067965
9      5       0.067136
13     4       0.063793
9      2       0.057959
12     5       0.049208
14     4       0.043783
11     1       0.043366
13     5       0.041283
       1       0.035031
15     2       0.035021
14     1       0.033360
10     1       0.028355
14     5       0.025856
12     1       0.022520
15     3       0.017925
16     2       0.012926
15     1       0.007921
9      1       0.007087
16     3       0.006257
15     4       0.005003
       5       0.004169
16     1       0.003753
       4       0.001668
       5       0.000833
11     0       0.000000
10  